# Part 1: FDIC Data Acquisition, Verification & Lakehouse Storage Architecture
This notebook verifies the reproducible acquisition of 170 quarters (1984Q1–2026Q2) of FDIC Call Report data, audits manifest completeness against API totals, and benchmarks our Hive-partitioned Zstandard Parquet Bronze layer against raw CSV and unpartitioned layouts.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import polars as pl
from src.bfews.quality import audit_manifest

manifest_check = audit_manifest()
print('Manifest Audit Summary:', manifest_check)

Manifest Audit Summary: {'quarters_downloaded': 170, 'total_manifest_rows': 1682616, 'incomplete_quarters': [], 'api_mismatch_quarters': []}


In [2]:
bench_df = pl.read_csv('../outputs/tables/part1_storage_benchmark.csv')
print(bench_df)
bronze_files = list(Path('../data/bronze/financials').rglob('*.parquet'))
print(f'Bronze partition count: {len(bronze_files)} files across {len({p.parent for p in bronze_files})} year folders')

shape: (6, 6)
┌─────────────────────┬─────────┬─────────────┬───────────────┬───────────────┬────────────────────┐
│ format              ┆ size_mb ┆ full_read_s ┆ read_3_cols_s ┆ filter_2023_s ┆ compression_vs_csv │
│ ---                 ┆ ---     ┆ ---         ┆ ---           ┆ ---           ┆ ---                │
│ str                 ┆ f64     ┆ f64         ┆ f64           ┆ f64           ┆ f64                │
╞═════════════════════╪═════════╪═════════════╪═══════════════╪═══════════════╪════════════════════╡
│ CSV (uncompressed)  ┆ 425.7   ┆ 0.754       ┆ 0.209         ┆ 0.623         ┆ 1.0                │
│ Parquet single      ┆ 210.8   ┆ 0.182       ┆ 0.034         ┆ 0.032         ┆ 2.02               │
│ file, Snappy        ┆         ┆             ┆               ┆               ┆                    │
│ Parquet single      ┆ 153.8   ┆ 0.205       ┆ 0.036         ┆ 0.032         ┆ 2.77               │
│ file, Zstd-3        ┆         ┆             ┆               ┆              